# Data tour: what is in each file

One snapshot (`head()`) of every dataset in the project, in pipeline order:

| # | Dataset | Made by | Where |
|---|---|---|---|
| 1 | Raw recording: MTA predictions | `logger.py` | `data/vm/<day>/predictions.csv.gz` (PC copy: `data/raw/`) |
| 2 | Raw recording: train positions | `logger.py` | `data/vm/<day>/vehicles.csv.gz` |
| 3 | History: actual stop times | `get_history.py` | `data/history/subwaydatanyc_<day>_csv.tar.xz` |
| 4 | Labels: true arrival times (answer key) | `labels.py` | `data/labels/arrivals.parquet` |
| 5 | Correction: history → platform time, per station | `correction.py` | `data/labels/correction.parquet` |
| 6 | Rows: model inputs + target | `rows.py` | `data/rows/{history,train,val}.parquet` |

All times in the files are Unix seconds; the `*_ny` columns added here show them as New York time. Sections 5–6 fill in once those steps are built.

In [1]:
import tarfile
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # the countdown/ folder
DATA = ROOT / 'data'
DAY = '2026-11-12'                                                         # any recorded day
pd.set_option('display.width', 200, 'display.max_columns', 40)

def ny(df, *cols):
    """Add a readable New York time column next to each Unix-seconds column."""
    for c in cols:
        df[c + '_ny'] = pd.to_datetime(df[c], unit='s', utc=True).dt.tz_convert('America/New_York').dt.strftime('%m-%d %H:%M:%S')
    return df

## 1. Raw recording: MTA predictions (`predictions.csv.gz`)

Every 30 s (`poll`), for every train, the MTA's predicted arrival (`arr`) and departure (`dep`) at **each stop it has not reached yet** (`idx` = 0 for the next stop, 1 for the one after, …). About 26 M rows per day. This is the countdown we try to beat (baseline B0).

In [2]:
pred = pd.read_csv(DATA / 'vm' / DAY / 'predictions.csv.gz', nrows=10)
ny(pred, 'poll', 'arr')

,poll,feed,feed_ts,trip_id,start_date,route_id,idx,stop_id,arr,dep,poll_ny,arr_ny
0,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,0,124S,1794459694,1794459694,11-12 00:00:14,11-12 00:01:34
1,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,1,125S,1794459784,1794459784,11-12 00:00:14,11-12 00:03:04
2,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,2,126S,1794459904,1794459904,11-12 00:00:14,11-12 00:05:04
3,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,3,127S,1794459994,1794460054,11-12 00:00:14,11-12 00:06:34
4,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,4,128S,1794460144,1794460144,11-12 00:00:14,11-12 00:09:04
5,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,5,129S,1794460234,1794460234,11-12 00:00:14,11-12 00:10:34
6,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,6,130S,1794460294,1794460294,11-12 00:00:14,11-12 00:11:34
7,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,7,131S,1794460354,1794460354,11-12 00:00:14,11-12 00:12:34
8,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,8,132S,1794460414,1794460414,11-12 00:00:14,11-12 00:13:34
9,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,9,133S,1794460504,1794460504,11-12 00:00:14,11-12 00:15:04


## 2. Raw recording: train positions (`vehicles.csv.gz`)

Every 30 s, one row per train: the stop it is at or heading to, and its `status`: **1 = STOPPED_AT** (standing at the platform), 2 = IN_TRANSIT_TO, 0 = INCOMING_AT. `veh_ts` is the train's own timestamp for that report. About 1.5 M rows per day.

In [3]:
veh = pd.read_csv(DATA / 'vm' / DAY / 'vehicles.csv.gz', nrows=10)
veh['status_name'] = veh.status.map({0: 'INCOMING_AT', 1: 'STOPPED_AT', 2: 'IN_TRANSIT_TO'})
ny(veh, 'poll', 'veh_ts')

,poll,feed,feed_ts,trip_id,start_date,route_id,stop_id,status,seq,veh_ts,status_name,poll_ny,veh_ts_ny
0,1794459614,gtfs,1794459610,283450_1..S15R,20261111,1,124S,2,22,1794459604,IN_TRANSIT_TO,11-12 00:00:14,11-12 00:00:04
1,1794459614,gtfs,1794459610,282900_1..N15R,20261111,1,106N,2,35,1794459609,IN_TRANSIT_TO,11-12 00:00:14,11-12 00:00:09
2,1794459614,gtfs,1794459610,283900_1..N15R,20261111,1,113N,1,28,1794459592,STOPPED_AT,11-12 00:00:14,11-11 23:59:52
3,1794459614,gtfs,1794459610,284650_1..S15R,20261111,1,119S,0,17,1794459600,INCOMING_AT,11-12 00:00:14,11-12 00:00:00
4,1794459614,gtfs,1794459610,284900_1..N15R,20261111,1,119N,1,22,1794459598,STOPPED_AT,11-12 00:00:14,11-11 23:59:58
5,1794459614,gtfs,1794459610,285850_1..S15R,20261111,1,116S,2,14,1794459608,IN_TRANSIT_TO,11-12 00:00:14,11-12 00:00:08
6,1794459614,gtfs,1794459610,285900_1..N15R,20261111,1,124N,1,17,1794459605,STOPPED_AT,11-12 00:00:14,11-12 00:00:05
7,1794459614,gtfs,1794459610,286900_1..N15R,20261111,1,131N,1,10,1794459595,STOPPED_AT,11-12 00:00:14,11-11 23:59:55
8,1794459614,gtfs,1794459610,287250_1..S15R,20261111,1,109S,2,7,1794459604,IN_TRANSIT_TO,11-12 00:00:14,11-12 00:00:04
9,1794459614,gtfs,1794459610,287900_1..N15R,20261111,1,139N,1,2,1794459607,STOPPED_AT,11-12 00:00:14,11-12 00:00:07


## 3. History: actual stop times (subwaydata.nyc)

One compressed file per day, May 1 – Nov 13, with two tables. `stop_times`: one row per train per stop, with when it arrived and departed ("last reported" definition: 30 s late on numbered lines, ~77 s on lettered lines). `trips`: one row per train, linking `trip_uid` to the line (`route_id`) and the MTA `trip_id`. About 230 K stop rows per weekday. **No MTA predictions here**, which is why we record.

In [ ]:
with tarfile.open(DATA / 'history' / f'subwaydatanyc_{DAY}_csv.tar.xz') as t:
    stop_times = pd.read_csv(t.extractfile(f'subwaydatanyc_{DAY}_stop_times.csv'))
    trips = pd.read_csv(t.extractfile(f'subwaydatanyc_{DAY}_trips.csv'))
print(f'{len(stop_times):,} stop rows, {len(trips):,} trips')
ny(stop_times.head(10).copy(), 'arrival_time', 'departure_time')

In [ ]:
trips.head(5)

## 4. Labels: true arrival times (`arrivals.parquet`)

Our answer key, built from section 2: for each train at each stop, `arrived` = the train's first STOPPED_AT report there, `left` = its last one, `known` = the first poll that showed it (when we could have known), `src` = which recorder saw it (vm, pc or both). Oct 31 – Nov 14, about 2.9 M arrivals.

In [ ]:
lab = pd.read_parquet(DATA / 'labels' / 'arrivals.parquet')
print(f'{len(lab):,} arrivals;', lab.src.value_counts(normalize=True).round(3).to_dict())
ny(lab.sample(10, random_state=0).copy(), 'arrived', 'known')

## 5. Correction (`correction.parquet`), step 3

Per station and direction: how many seconds the history's times run late compared with the platform (STOPPED_AT) time. Subtracted from history before training.

In [ ]:
f = DATA / 'labels' / 'correction.parquet'
pd.read_parquet(f).head(10) if f.exists() else print('not built yet (step 3)')

## 6. Rows: what the models train on (`data/rows/`), step 4

One row = one train × one target stop × one prediction moment `p`: the features in README §2, the target `y` (actual minus typical travel time), and for live rows the MTA's prediction (for scoring only). `history.parquet` = pretraining, `train.parquet` = Oct 31 – Nov 7, `val.parquet` = Nov 8 – 14.

In [ ]:
for name in ['history', 'train', 'val']:
    f = DATA / 'rows' / f'{name}.parquet'
    if f.exists():
        rows = pd.read_parquet(f)
        print(f'{name}: {len(rows):,} rows')
        display(rows.head(5))
    else:
        print(f'{name}: not built yet (step 4)')